# Competitive Negative-Feedback Adaptation

This notebook is the SI-style computation for the adaptation part of the paper.  The workflow follows the regime-based R-index calculation:

1. enumerate dominance regimes of the competitive binding network;
2. keep full-dimensional regimes of the coupled binding-catalysis system;
3. impose flux balance and local stability;
4. impose steady-state invariance with respect to input `tI`;
5. impose input responsiveness;
6. compute the R-index as the asymptotic volume of the remaining regime cones.

Symbols use the paper notation internally: `Symbol("A⋆")`, `Symbol("B⋆")`, `Symbol("tA⋆")`, `Symbol("tB⋆")`, `Symbol("tÃ")`, and `Symbol("tB̃")`.

In [ ]:
begin
    using Pkg
    Pkg.activate(@__DIR__)
end

using LinearAlgebra
using SparseArrays
using CairoMakie
using GraphMakie
using Makie
using Polyhedra
using BindingAndCatalysis

CairoMakie.activate!(type="png")

const A_ACTIVE = Symbol("A⋆")
const B_ACTIVE = Symbol("B⋆")
const tA_ACTIVE = Symbol("tA⋆")
const tB_ACTIVE = Symbol("tB⋆")
const tA_TOTAL = Symbol("tÃ")
const tB_TOTAL = Symbol("tB̃")

const DEFAULT_RESPONSE_THRESHOLD = 0.1
const DEFAULT_VOLUME_REL_TOL = 0.01
const DEFAULT_VOLUME_TIME_LIMIT = 10.0 # seconds， to prevent long computations
const INVARIANCE_ATOL = 1e-6

## Model

The competitive binding network is

`I + A ⇌ C1`, `A⋆ + B⋆ ⇌ C2`, `A⋆ + B ⇌ C3`, and `E + B⋆ ⇌ C4`.

The slow variables are `tA⋆` and `tB⋆`, with dynamics

`d tA⋆/dt = kA1 C1 - kA2 C2`,
`d tB⋆/dt = kB1 C3 - kB2 C4`.

The conserved totals are `tÃ`, `tB̃`, `tI`, `tE`, `tA⋆`, and `tB⋆` after the package reorders catalysis-involving quantities.

In [ ]:
function build_competitive_adaptation_model()
    x_sym = [
        :I, :A, A_ACTIVE, :B, B_ACTIVE, :E,
        :C1, :C2, :C3, :C4,
    ]

    q_sym = [
        :tI,
        :tA, tA_ACTIVE,
        :tB, tB_ACTIVE, :tE,
    ]

    K_sym = [:KA1, :KA2, :KB1, :KB2]

    # Binding equilibria:
    # KA1: I + A      <-> C1
    # KA2: A⋆ + B⋆   <-> C2
    # KB1: A⋆ + B    <-> C3
    # KB2: E + B⋆    <-> C4
    N = [
        1 1 0 0 0 0 -1  0  0  0
        0 0 1 0 1 0  0 -1  0  0
        0 0 1 1 0 0  0  0 -1  0
        0 0 0 0 1 1  0  0  0 -1
    ]

    model = Bnc(N=N, x_sym=x_sym, q_sym=q_sym, K_sym=K_sym)

    # Fluxes:
    # v1 = kA1*C1 : A  -> A⋆
    # v2 = kA2*C2 : A⋆ -> A
    # v3 = kB1*C3 : B  -> B⋆
    # v4 = kB2*C4 : B⋆ -> B
    Π = diagm(ones(Int, 4))
    Γ = [
         1 -1  0  0   # tA
        -1  1  0  0   # tA⋆
         0  0  1 -1   # tB
         0  0 -1  1   # tB⋆
    ]

    update_catalysis!(
        model;
        Π,
        Γ,
        x_picked=[:C1, :C2, :C3, :C4],
        q_picked=[tA_ACTIVE, :tA, tB_ACTIVE, :tB],
        w_sym=[tA_TOTAL, tB_TOTAL],
        k_sym=[:kA1, :kA2, :kB1, :kB2],
    )

    return model
end

adaptation_model = build_competitive_adaptation_model()

In [ ]:
show_conservation(adaptation_model)

In [ ]:
show_equilibrium(adaptation_model)

In [ ]:
show_catalysis_dynamics(adaptation_model)

## Adaptation Criteria

Steady-state invariance means that the steady-state expression for the output has zero logarithmic derivative with respect to `tI`:

- for free active output `A⋆`, use the BNC steady-state expression `A⋆(w,K,k)` and require `∂ log A⋆ / ∂ log tI = 0`;
- for total active output `tA⋆`, use the steady-state `q_cat` expression `tA⋆(w,K,k)` and require `∂ log tA⋆ / ∂ log tI = 0`.

Input responsiveness is evaluated immediately after an input perturbation under binding equilibrium.  For `tA⋆`, the relevant fluxes are

`f⁺ = kA1 C1` and `f⁻ = kA2 C2`.

The original condition is therefore

`∂ log f⁺ / ∂ log tI - ∂ log f⁻ / ∂ log tI > threshold`.

The earlier exploratory code used an equivalent shortcut in these valid regimes because `C2` has zero `tI` log-order there.  The code below computes the original two-flux condition explicitly.  For free `A⋆`, responsiveness additionally requires

`∂ log A⋆ / ∂ log tA⋆ > threshold`

inside the binding regime.  The default threshold is `0.1`; it avoids treating numerically singular or nearly-flat responses as genuine responsiveness.

In [ ]:
function binding_order_matrix(rgm)
    binding_rgm = get_binding_regime(rgm)
    is_singular(binding_rgm) ? get_H_numerically(binding_rgm) : get_H(binding_rgm)
end

function binding_log_order(rgm, output_sym::Symbol, input_sym::Symbol)
    model = get_binding_network(rgm)
    H = binding_order_matrix(rgm)
    return H[locate_sym_x(model, output_sym), locate_sym_qK(model, input_sym)]
end

function steady_state_invariance(rgm; atol=INVARIANCE_ATOL)
    model = get_binding_network(rgm)
    tI_idx = locate_sym_wKk(model, :tI)
    active_idx = locate_sym_x(model, A_ACTIVE)
    total_active_idx = locate_sym_qcat(model, tA_ACTIVE)

    H = get_H(rgm)
    F, _ = get_qcat_F_F0(rgm)

    dlog_A_dlog_tI = H[active_idx, tI_idx]
    dlog_tA_dlog_tI = F[total_active_idx, tI_idx]

    return (;
        A_active = abs(dlog_A_dlog_tI) <= atol,
        tA_active = abs(dlog_tA_dlog_tI) <= atol,
        dlog_A_active_dlog_tI = dlog_A_dlog_tI,
        dlog_tA_active_dlog_tI = dlog_tA_dlog_tI,
    )
end

function input_responsiveness(rgm; threshold=DEFAULT_RESPONSE_THRESHOLD)
    dlog_f_plus_dlog_tI = binding_log_order(rgm, :C1, :tI)
    dlog_f_minus_dlog_tI = binding_log_order(rgm, :C2, :tI)
    flux_drive = dlog_f_plus_dlog_tI - dlog_f_minus_dlog_tI
    active_gate = binding_log_order(rgm, A_ACTIVE, tA_ACTIVE)

    total_active_responsive = flux_drive > threshold
    active_responsive = total_active_responsive && active_gate > threshold

    return (;
        A_active = active_responsive,
        tA_active = total_active_responsive,
        flux_drive,
        dlog_f_plus_dlog_tI,
        dlog_f_minus_dlog_tI,
        dlog_A_active_dlog_tA_active = active_gate,
    )
end

function adaptation_signature(rgm; threshold=DEFAULT_RESPONSE_THRESHOLD)
    invariance = steady_state_invariance(rgm)
    response = input_responsiveness(rgm; threshold)
    return (;
        A_active = invariance.A_active && response.A_active,
        tA_active = invariance.tA_active && response.tA_active,
        invariance,
        response,
    )
end

function find_adaptation_regimes(model; threshold=DEFAULT_RESPONSE_THRESHOLD)
    full_dimensional = get_bnc_regimes(model; singular=false)
    stable = filter(is_stable, full_dimensional)

    valid_A_active = filter(stable) do rgm
        adaptation_signature(rgm; threshold).A_active
    end

    valid_tA_active = filter(stable) do rgm
        adaptation_signature(rgm; threshold).tA_active
    end

    shared = intersect(valid_A_active, valid_tA_active)
    union_valid = union(valid_A_active, valid_tA_active)

    return (;
        full_dimensional,
        stable,
        valid_A_active,
        valid_tA_active,
        shared,
        union_valid,
    )
end

## Regime Classification

The counts below reproduce the adaptation classification: competitive binding does not eliminate adaptation.  Some accepted regimes have singular binding sub-regimes; those are handled by numerical binding log-orders in the responsiveness tests, while the coupled BNC regime remains full-dimensional in `(w,K,k)`.

In [ ]:
adaptation_sets = find_adaptation_regimes(adaptation_model; threshold=DEFAULT_RESPONSE_THRESHOLD)

regime_count_summary = (;
    all_binding_dominance_regimes = n_bnc_regimes(adaptation_model),
    full_dimensional_BNC_regimes = length(adaptation_sets.full_dimensional),
    stable_full_dimensional_BNC_regimes = length(adaptation_sets.stable),
    A_active_adaptation = length(adaptation_sets.valid_A_active),
    tA_active_adaptation = length(adaptation_sets.valid_tA_active),
    shared_outputs = length(adaptation_sets.shared),
    singular_binding_in_A_active = count(r -> is_singular(get_binding_regime(r)), adaptation_sets.valid_A_active),
    singular_binding_in_tA_active = count(r -> is_singular(get_binding_regime(r)), adaptation_sets.valid_tA_active),
)

In [ ]:
function regime_signature_table(rgms; threshold=DEFAULT_RESPONSE_THRESHOLD)
    map(rgms) do rgm
        sig = adaptation_signature(rgm; threshold)
        (;
            idx = get_idx(rgm),
            binding_perm = get_binding_perm(rgm),
            catalysis_perm = get_perm(get_catalysis_regime(rgm)),
            binding_singular = is_singular(get_binding_regime(rgm)),
            stable = is_stable(rgm),
            A_active = sig.A_active,
            tA_active = sig.tA_active,
            dlog_A_active_dlog_tI = sig.invariance.dlog_A_active_dlog_tI,
            dlog_tA_active_dlog_tI = sig.invariance.dlog_tA_active_dlog_tI,
            flux_drive = sig.response.flux_drive,
            dlog_A_active_dlog_tA_active = sig.response.dlog_A_active_dlog_tA_active,
        )
    end
end

valid_regime_table = regime_signature_table(adaptation_sets.union_valid)

## R-Index Estimate

The regime cones are disjoint cells of the same parameter space, so the R-index for a set of accepted regimes is the sum of their asymptotic volumes.  The tolerance below is intentionally moderate for a reproducible notebook run; the paper reports tighter estimates for the final table.

In [ ]:
adaptation_volume_estimate = let
    rgms = adaptation_sets.union_valid
    volumes = get_volumes(
        rgms;
        reltol=DEFAULT_VOLUME_REL_TOL,
        time_limit=DEFAULT_VOLUME_TIME_LIMIT,
    )

    volume_of(selection) = sum(volumes[findfirst(==(rgm), rgms)] for rgm in selection)

    (;
        per_regime = [(idx=get_idx(rgms[i]), volume=volumes[i]) for i in eachindex(rgms)],
        R_A_active = volume_of(adaptation_sets.valid_A_active),
        R_tA_active = volume_of(adaptation_sets.valid_tA_active),
        R_shared = volume_of(adaptation_sets.shared),
    )
end

## Example Regime

The paper highlights the largest shared regime, index `28` in the current ordering.  Its dominance relations include

`tA⋆ ≈ C3`, `tB⋆ ≈ B⋆`, `tÃ ≈ A`, `tB̃ ≈ B`, `tI ≈ C1`, and `tE ≈ C4`.

The reduced dynamics have the same integral-feedback structure as the classical analysis:

`d tB⋆/dt = kB1 tA⋆ - kB2 tE`, hence `tA⋆ = kB2 tE/kB1` at steady state, independent of `tI`.  The free active output in this regime is also independent of `tI`:

`A⋆ = KB1 kB2 tE/(kB1 tB̃)`.

In [ ]:
example_rgm = get_bnc_regime(adaptation_model, 28)
example_signature = adaptation_signature(example_rgm)

example_overview = (;
    regime = example_rgm,
    signature = example_signature,
    assigned_outputs = (
        A_active = example_rgm in adaptation_sets.valid_A_active,
        tA_active = example_rgm in adaptation_sets.valid_tA_active,
    ),
)

In [ ]:
show_dominant_condition(get_binding_regime(example_rgm))

In [ ]:
show_condition_wKk(example_rgm)

In [ ]:
show_expression_qcat(example_rgm)

In [ ]:
show_expression_x(example_rgm)

In [ ]:
show_catalysis_dynamics(example_rgm)

## Numerical Adaptation Check

The point below uses the parameter values reported for the simulation figure in the paper:

`log10 tÃ = 2.9`, `log10 tB̃ = 5.8`, `log10 tE = -1.3`, `log10 KA1 = -1.1`, `log10 KA2 = 2.9`, `log10 KB1 = 2.9`, `log10 KB2 = 0`, `log10 kA1 = -3.4`, `log10 kA2 = 0`, `log10 kB1 = -2.2`, `log10 kB2 = 0`.

The input starts at `log10 tI = 1.1`, steps to `2.1`, and then to `0.1`.  The chosen point is assigned to regime `28`, and the simulation shows transient responses followed by recovery of both `A⋆` and `tA⋆` to their setpoints.

In [ ]:
function log_parameter_vector(model, assignments::Dict{Symbol, <:Real})
    p = zeros(Float64, length(wKk_symbol(model)))
    for (sym, val) in assignments
        p[locate_sym_wKk(model, sym)] = Float64(val)
    end
    return p
end

paper_logwKk = Dict{Symbol, Float64}(
    tA_TOTAL => 2.9,
    tB_TOTAL => 5.8,
    :tI => 1.1,
    :tE => -1.3,
    :KA1 => -1.1,
    :KA2 => 2.9,
    :KB1 => 2.9,
    :KB2 => 0.0,
    :kA1 => -3.4,
    :kA2 => 0.0,
    :kB1 => -2.2,
    :kB2 => 0.0,
)

paper_p = log_parameter_vector(adaptation_model, paper_logwKk)
assigned_example_idx = assign_bnc_regime_wKk(adaptation_model, paper_p)
@assert assigned_example_idx == get_idx(example_rgm)

F_example, F0_example = get_qcat_F_F0(example_rgm)
H_example, H0_example = get_H_H0(example_rgm)
logqcat0_example = Vector{Float64}(F_example * paper_p .+ F0_example)
logx0_example = Vector{Float64}(H_example * paper_p .+ H0_example)

example_setpoints = (;
    log_A_active = logx0_example[locate_sym_x(adaptation_model, A_ACTIVE)],
    log_tA_active = logqcat0_example[locate_sym_qcat(adaptation_model, tA_ACTIVE)],
)

In [ ]:
stop1 = 8_0000.0
stop2 = 25_0000.0
endt = 60_0000.0
input_start = paper_p[locate_sym_wKk(adaptation_model, :tI)]
logtI_profile(t) = t < stop1 ? input_start : t < stop2 ? 2.1 : 0.1

savegrid = range(0.0, endt, length=450)

dyn_A_active = simulate_adaptation(
    adaptation_model;
    p=paper_p,
    logtI=logtI_profile,
    tspan=(0.0, endt),
    observe=A_ACTIVE,
    logqcat0=logqcat0_example,
    saveat=savegrid,
    tstops=[stop1, stop2],
    maxiters=300_000,
)

tA_active_idx = locate_sym_qcat(adaptation_model, tA_ACTIVE)
log_tA_active_traj = vec(dyn_A_active.logqcat[tA_active_idx, :])

simulation_summary = (;
    assigned_example_idx,
    initial_log_A_active = dyn_A_active.logobserve[1],
    final_log_A_active = dyn_A_active.logobserve[end],
    setpoint_log_A_active = example_setpoints.log_A_active,
    final_A_active_error = dyn_A_active.logobserve[end] - example_setpoints.log_A_active,
    initial_log_tA_active = log_tA_active_traj[1],
    final_log_tA_active = log_tA_active_traj[end],
    setpoint_log_tA_active = example_setpoints.log_tA_active,
    final_tA_active_error = log_tA_active_traj[end] - example_setpoints.log_tA_active,
)

In [ ]:
let
    fig = Figure(size=(820, 680))

    ax_input = Axis(fig[1, 1], ylabel="log10 tI")
    lines!(ax_input, dyn_A_active.t, dyn_A_active.logtI; color=:black, linewidth=2)
    hidexdecorations!(ax_input; grid=false)

    ax_tA = Axis(fig[2, 1], ylabel="log10 tA⋆")
    lines!(ax_tA, dyn_A_active.t, log_tA_active_traj; color=:dodgerblue4, linewidth=2)
    hlines!(ax_tA, example_setpoints.log_tA_active; color=:firebrick, linestyle=:dash, linewidth=2)
    hidexdecorations!(ax_tA; grid=false)

    ax_A = Axis(fig[3, 1], xlabel="time", ylabel="log10 A⋆")
    lines!(ax_A, dyn_A_active.t, dyn_A_active.logobserve; color=:darkgreen, linewidth=2)
    hlines!(ax_A, example_setpoints.log_A_active; color=:firebrick, linestyle=:dash, linewidth=2)

    linkxaxes!(ax_input, ax_tA, ax_A)
    fig
end